
# Función de transferencia: posición de la carga vs voltaje de entrada

Este notebook deriva **paso a paso** la función de transferencia linealizada que relaciona la **posición angular de la carga** \(\theta_l(t)\) con el **voltaje de entrada** \(V(t)\) de un motor DC con transmisión \(n\).

## 1) Ecuaciones del enunciado

### Dinámica de la carga
\[
\tau_L = I\,\ddot{\theta}_l + m\left(l^2\ddot{\theta}_l - l g \cos(\theta_l)\right)
\]
equivalentemente:
\[
\tau_L = (I + m l^2)\ddot{\theta}_l - m l g \cos(\theta_l)
\]

### Modelo del motor DC
\[
L\frac{di_a}{dt} + R i_a = V - V_b,\qquad
\tau_m = k_m i_a,\qquad
V_b = k_b \dot{\theta}_m
\]
\[
J_m \ddot{\theta}_m + B\dot{\theta}_m = \tau_m - \frac{\tau_L}{n},\qquad
\theta_m = n\,\theta_l
\]

> Nota: el término no lineal está en \(\cos(\theta_l)\), por lo que se requiere linealización para obtener una función de transferencia.



## 2) Definición simbólica

Vamos a derivar \(\Theta_l(s)/V(s)\) en Laplace usando SymPy.


In [ ]:

import sympy as sp

# Variable de Laplace
s = sp.symbols('s')

# Parámetros (todos positivos en operación normal)
I, m, l, g = sp.symbols('I m l g', positive=True, real=True)       # carga
Jm, B = sp.symbols('Jm B', positive=True, real=True)               # motor mecánico
L, R = sp.symbols('L R', positive=True, real=True)                 # motor eléctrico
km, kb = sp.symbols('k_m k_b', positive=True, real=True)           # constantes
n = sp.symbols('n', positive=True, real=True)                      # relación de transmisión

# Punto de operación para linealizar
theta0 = sp.symbols('theta0', real=True)



## 3) Linealización de \(\cos(\theta_l)\)

Se linealiza alrededor de un punto de operación:
\[
\theta_l(t) = \theta_0 + \delta\theta(t)
\]

Usamos expansión de primer orden:
\[
\cos(\theta_0 + \delta\theta) \approx \cos(\theta_0) - \sin(\theta_0)\,\delta\theta
\]

Como
\[
\tau_L = (I+m l^2)\ddot{\theta}_l - m l g \cos(\theta_l),
\]
al separar el equilibrio (\(\ddot{\theta}_0=0\)) y quedarnos con pequeñas señales, el **incremento** de torque queda:

\[
\delta\tau_L = (I+m l^2)\,\delta\ddot{\theta} + (m l g\sin\theta_0)\,\delta\theta
\]

Definimos:
\[
J_L = I + m l^2,\qquad K_g = m l g \sin\theta_0
\]
Entonces:
\[
\delta\tau_L = J_L\,\delta\ddot{\theta} + K_g\,\delta\theta
\]

> Importante: el término constante \(-m l g\cos\theta_0\) se **absorbe** en el punto de operación (corriente/torque DC) y no aparece en la función de transferencia incremental.


In [ ]:

JL = I + m*l**2
Kg = m*l*g*sp.sin(theta0)

JL, Kg



## 4) Combinar carga + motor + transmisión (modelo lineal incremental)

Relaciones incrementales (en Laplace, usando \(\Theta(s)\equiv \Delta\Theta_l(s)\)):

- Cinemática: \(\theta_m = n\theta_l \Rightarrow \Omega_m(s)=n\,s\,\Theta(s)\), \(\ddot{\theta}_m \Rightarrow n s^2 \Theta(s)\).
- Torque de carga: \(\Tau_L(s) = (J_L s^2 + K_g)\Theta(s)\).
- Mecánica del motor:
\[
J_m \ddot{\theta}_m + B\dot{\theta}_m = k_m i_a - \frac{\tau_L}{n}
\]
- Eléctrica:
\[
(Ls+R)I_a(s) = V(s) - k_b\,\Omega_m(s)
\]

A partir de estas ecuaciones, eliminamos \(I_a(s)\) y obtenemos \(\Theta(s)/V(s)\).


In [ ]:

# Definimos variables de Laplace
Theta = sp.symbols('Theta')  # representará Θ(s), solo como marcador
Ia = sp.symbols('Ia')        # I_a(s)
V = sp.symbols('V')          # V(s)

# Carga: TauL(s) = (JL*s^2 + Kg)*Theta
TauL = (JL*s**2 + Kg)*Theta

# Motor: omega_m = n*s*Theta, alpha_m = n*s^2*Theta
omega_m = n*s*Theta
alpha_m = n*s**2*Theta

# Ecuación mecánica: Jm*alpha_m + B*omega_m = km*Ia - TauL/n
eq_mech = sp.Eq(Jm*alpha_m + B*omega_m, km*Ia - TauL/n)

# Ecuación eléctrica: (L*s + R)*Ia = V - kb*omega_m
eq_elec = sp.Eq((L*s + R)*Ia, V - kb*omega_m)

eq_mech, eq_elec



### 4.1) Eliminar \(I_a(s)\)

De la ecuación mecánica:
\[
k_m I_a(s)= \left(J_{eq}s^2 + B_{eq}s + K_g\right)\Theta(s)
\]
donde
\[
J_{eq}=J_m n^2 + J_L,\qquad B_{eq}=B n^2
\]

Luego se sustituye en la ecuación eléctrica para obtener \(\Theta(s)/V(s)\).


In [ ]:

Jeq = Jm*n**2 + JL
Beq = B*n**2

# Resolver Ia desde la mecánica y sustituir en la eléctrica
Ia_expr = sp.solve(eq_mech, Ia)[0]
Ia_expr_simpl = sp.simplify(Ia_expr.subs({TauL: (JL*s**2 + Kg)*Theta}))

Ia_expr_simpl


In [ ]:

# Sustituir Ia en la ecuación eléctrica y despejar Theta/V
eq_elec_sub = eq_elec.subs(Ia, Ia_expr_simpl)
eq_elec_sub_simpl = sp.simplify(eq_elec_sub)

eq_elec_sub_simpl


In [ ]:

# Despejar Theta en función de V: Theta = G(s)*V
Theta_over_V = sp.simplify(sp.solve(eq_elec_sub_simpl, Theta)[0] / V)
Theta_over_V



## 5) Función de transferencia final (forma compacta)

El resultado linealizado incremental es:

\[
\boxed{
\frac{\Theta_l(s)}{V(s)} =
\frac{k_m\,n}{
(Ls+R)\left(J_{eq}s^2 + B_{eq}s + K_g\right) + k_m k_b n^2 s
}}
\]

con
\[
J_{eq}=J_m n^2 + (I+m l^2),\qquad
B_{eq}=B n^2,\qquad
K_g = m l g \sin(\theta_0).
\]

A continuación la expandimos como polinomios (numerador/denominador) para usarla en simulación.


In [ ]:

# Expansión polinómica numerador/denominador
G = sp.together(Theta_over_V)  # forma racional
num, den = sp.fraction(G)

num = sp.expand(num)
den = sp.expand(den)

num, den



## 6) Coeficientes del denominador en potencias de \(s\)

Esto ayuda a pasar a `scipy.signal.TransferFunction`.


In [ ]:

den_poly = sp.Poly(den, s)
num_poly = sp.Poly(num, s)

den_poly.all_coeffs(), num_poly.all_coeffs()



## 7) (Opcional) Ejemplo numérico y respuesta al escalón

Rellena **tus parámetros** y simula.  
Si no conoces \(\theta_0\), una elección común es:
- \(\theta_0=0\)  \(\Rightarrow K_g = 0\) (la gravedad se absorbe en el equilibrio y no aporta rigidez incremental).
- \(\theta_0=\pi/2\) \(\Rightarrow K_g = mlg\) (máxima rigidez incremental).

**Importante:** el modelo es de pequeñas señales (\(\delta\theta\)), así que la excitación debe ser suficientemente pequeña para que la aproximación sea válida.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# ====== EJEMPLO (reemplaza por tus valores reales) ======
params_example = {
    I: 0.01,    # kg·m^2
    m: 1.0,     # kg
    l: 0.2,     # m
    g: 9.81,    # m/s^2
    Jm: 1e-4,   # kg·m^2
    B: 1e-3,    # N·m·s/rad
    L: 1e-3,    # H
    R: 1.0,     # ohm
    km: 0.1,    # N·m/A
    kb: 0.1,    # V·s/rad
    n: 10.0,    # -
    theta0: 0.0 # rad (elige tu punto de operación)
}

# Sustituir parámetros en numerador y denominador
num_num = sp.N(num.subs(params_example))
den_num = sp.N(den.subs(params_example))

# Convertir a arreglos float para SciPy (coeficientes descendentes en s)
num_coeffs = [float(c) for c in sp.Poly(num_num, s).all_coeffs()]
den_coeffs = [float(c) for c in sp.Poly(den_num, s).all_coeffs()]

sys = signal.TransferFunction(num_coeffs, den_coeffs)

# Respuesta al escalón (1 V incremental)
t, y = signal.step(sys)

plt.figure()
plt.plot(t, y)
plt.xlabel("t [s]")
plt.ylabel(r"$\delta \\,\\theta_l$ [rad] por 1 V")
plt.title("Respuesta al escalón (modelo linealizado)")
plt.grid(True)
plt.show()

num_coeffs, den_coeffs
